# **FLIGHT PRICE PREDICTION — LINEAR REGRESSION**
Dataset: `Dataset.csv` (flight tickets between 6 Indian cities).

The user enters **boarding point, dropping point, time of flight** (Early_Morning / Morning / Afternoon / Evening / Night / Late_Night), **class** and **how many days before** the flight they are booking. The trained model then **predicts the ticket price** and shows graphs.

Only concepts from **Practice Lab 7 (Linear Regression)** are used: pandas, numpy, matplotlib, `train_test_split`, `Pipeline` + `SimpleImputer` + `OneHotEncoder` + `ColumnTransformer`, `LinearRegression`, and MAE / MSE / RMSE / R².

$$\hat{y} = b_0 + b_1x_1 + b_2x_2 + \cdots + b_nx_n$$
$\hat{y}$ = predicted price; $x_1, x_2, \ldots$ = source city, destination city, departure time, class, days left.

### 1. Import Libraries

In [ ]:
import pandas as pd                  # Read and work with tabular data
import numpy as np                   # Numerical operations
import matplotlib.pyplot as plt      # Graphs and visualizations

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Display graphs inside the notebook
%matplotlib inline

print("Libraries imported successfully!")

### 2. Read Dataset

In [ ]:
df = pd.read_csv('Dataset.csv')   # change the path if your file is elsewhere
print("Dataset shape (rows, columns):", df.shape)
df.head()

### 3. Understand the dataset
- `shape`, `head()`, `tail()`, `info()`, `describe()`, `isnull().sum()`

In [ ]:
print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nLast ten rows:")
display(df.tail(10))

print("\nData types and missing-value overview:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nSummary of numeric columns:")
display(df.describe())

### 4. Explore the data with graphs
Histogram of price, scatter plot of days left vs price, and a bar chart of the average price for each departure time.

In [ ]:
# Histogram of the target column
plt.figure(figsize=(8, 4))
plt.hist(df["price"].dropna(), bins=30, edgecolor="black")
plt.title("Distribution of price")
plt.xlabel("price")
plt.ylabel("Number of flights")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# Scatter plot: days_left vs price (a random sample keeps the plot readable)
sample = df.sample(n=min(3000, len(df)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(sample["days_left"], sample["price"], alpha=0.35)
plt.title("price vs days_left")
plt.xlabel("days_left (days before the flight)")
plt.ylabel("price")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# Bar chart: average price for each departure time
avg_by_time = df.groupby("departure_time")["price"].mean().sort_values()

plt.figure(figsize=(9, 4))
plt.bar(avg_by_time.index, avg_by_time.values, edgecolor="black")
plt.title("Average price by departure time")
plt.xlabel("Departure time")
plt.ylabel("Average price")
plt.xticks(rotation=20)
plt.show()

### 5. Preprocess the data
1. Remove duplicate rows.
2. Select the target (`price`) and the input features — the same things the user will enter later.
3. Fill missing numeric values with the median, missing categories with the most frequent value.
4. One-hot encode the categorical columns.

The target is **not** included in `feature_columns` (no leakage).

In [ ]:
df = df.drop_duplicates().copy()

print("Available columns:", df.columns.tolist())

# Target (what we want to predict)
target_column = "price"

# Input features = what the user will enter
feature_columns = [
    "source_city", "destination_city", "departure_time",
    "class", "days_left"
]

# Validate column names
missing_columns = [
    col for col in [target_column] + feature_columns
    if col not in df.columns
]

if missing_columns:
    print("Error: These columns do not exist:", missing_columns)
else:
    print("Target:", target_column)
    print("Features:", feature_columns)
    display(df[[target_column] + feature_columns].head())

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Keep only selected features and target
model_df = df[feature_columns + [target_column]].copy()

# Convert the target to numeric; non-numeric values become NaN
model_df[target_column] = pd.to_numeric(model_df[target_column], errors="coerce")

# Remove rows where the target price is missing or invalid
model_df = model_df.dropna(subset=[target_column])

X = model_df[feature_columns]
y = model_df[target_column]

# Detect numeric and categorical input columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

# Fill missing numeric values with median
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Fill missing categories with the most frequent category, then encode them
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the appropriate preprocessing to each column type
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Rows available for modelling:", len(model_df))

### 6. Split into training and testing data (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

### 7. Build and train the Linear Regression model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model training completed!")

### 8. Make predictions on the test set

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": y_pred,
    "Error (Actual - Predicted)": y_test.to_numpy() - y_pred
})

display(comparison.head(10))

### 9. Evaluate the model
MAE, MSE, RMSE (same units as price, lower is better) and R² (closer to 1 is better).

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:,.3f}")
print(f"MSE  : {mse:,.3f}")
print(f"RMSE : {rmse:,.3f}")
print(f"R²   : {r2:.3f}")

### 10. Actual vs predicted prices

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, edgecolor="black")

# Reference line from the smallest to largest value
low = min(float(np.min(y_test)), float(np.min(y_pred)))
high = max(float(np.max(y_test)), float(np.max(y_pred)))
plt.plot([low, high], [low, high], linestyle="--", color="red", label="Perfect prediction")

plt.xlabel("Actual flight price")
plt.ylabel("Predicted flight price")
plt.title("Actual vs Predicted Flight Prices")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### 11. Five largest prediction errors

In [ ]:
comparison["Absolute Error"] = comparison["Error (Actual - Predicted)"].abs()
display(comparison.nlargest(5, "Absolute Error"))

### 12. Predict the price for the user's flight
The user chooses the **boarding point**, **dropping point**, **time of flight**, **class** and **how many days before** the flight they are booking. Valid options are taken from the dataset itself, and the input is checked before predicting.

In [ ]:
# Valid options (taken from the dataset)
cities = sorted(df["source_city"].unique())
times = list(df["departure_time"].unique())
classes = list(df["class"].unique())

print("Cities        :", cities)
print("Time of flight:", times)
print("Class         :", classes)

def ask(question, options):
    # Keep asking until the user types one of the valid options
    while True:
        answer = input(question + " " + str(options) + ": ").strip()
        for option in options:
            if answer.lower() == option.lower():
                return option
        print("Invalid choice, please type one of the options shown.")

user_source = ask("Boarding point", cities)
user_dest = ask("Dropping point", cities)
while user_dest == user_source:
    print("Boarding and dropping point cannot be the same.")
    user_dest = ask("Dropping point", cities)

user_time = ask("Time of flight", times)
user_class = ask("Class", classes)

while True:
    days_text = input("Days before the flight you are booking (1 to 49): ").strip()
    if days_text.isdigit() and 1 <= int(days_text) <= 49:
        user_days = int(days_text)
        break
    print("Please enter a whole number between 1 and 49.")

# One row with the SAME feature columns used during training
user_flight = pd.DataFrame({
    "source_city": [user_source],
    "destination_city": [user_dest],
    "departure_time": [user_time],
    "class": [user_class],
    "days_left": [user_days]
})

predicted_price = model.predict(user_flight)[0]

print("\n----- FLIGHT DETAILS -----")
print("From          :", user_source)
print("To            :", user_dest)
print("Time of flight:", user_time)
print("Class         :", user_class)
print("Booking       :", user_days, "days before the flight")
print(f"\nPredicted Flight Price: Rs. {predicted_price:,.0f}")

### 13. Graphs for the user's flight
1. How the predicted price changes with the number of days before booking.
2. The predicted price for each time of flight on the same route.

In [ ]:
# Graph 1: predicted price for 1 to 49 days before the flight
days_range = np.arange(1, 50)

days_df = pd.DataFrame({
    "source_city": user_source,
    "destination_city": user_dest,
    "departure_time": user_time,
    "class": user_class,
    "days_left": days_range
})
days_pred = model.predict(days_df)

plt.figure(figsize=(9, 5))
plt.plot(days_range, days_pred, marker="o", markersize=4, label="Predicted price")
plt.scatter([user_days], [predicted_price], color="red", s=90, zorder=5, label="Your booking")
plt.title(f"Predicted price vs days before flight ({user_source} to {user_dest}, {user_class})")
plt.xlabel("Days before the flight")
plt.ylabel("Predicted price")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# Graph 2: predicted price for every time of flight (same route, class and days left)
time_df = pd.DataFrame({
    "source_city": user_source,
    "destination_city": user_dest,
    "departure_time": times,
    "class": user_class,
    "days_left": user_days
})
time_pred = model.predict(time_df)

plt.figure(figsize=(9, 4))
plt.bar(times, time_pred, edgecolor="black")
plt.title(f"Predicted price by time of flight ({user_source} to {user_dest}, {user_days} days before)")
plt.xlabel("Time of flight")
plt.ylabel("Predicted price")
plt.xticks(rotation=20)
plt.show()

cheapest = times[int(np.argmin(time_pred))]
print("Cheapest time of flight for this route:", cheapest)